In [2]:
# Cell 1 — Imports & description
import os
import re
from glob import glob
from typing import List, Tuple

import numpy as np
import torch
import rasterio
from rasterio.windows import Window
import segmentation_models_pytorch as smp

"""
Batches aligned 2030 SSP rasters by internal blocks, normalizes per channel,
runs the loaded model (optionally with sigmoid), and writes compressed
single-band CISI GeoTIFFs per SSP.
"""


'\nBatches aligned 2030 SSP rasters by internal blocks, normalizes per channel,\nruns the loaded model (optionally with sigmoid), and writes compressed\nsingle-band CISI GeoTIFFs per SSP.\n'

In [3]:
# Cell 2 — Global config

# Root folder containing SSP1..SSP5 subfolders
DATA_ROOT = "READY_data"

# Where to write CISI projections
OUT_DIR   = "predictions"

# Trained model checkpoint (from training notebook)
CKPT_PATH = "checkpoints/best_unet_regression.pt"

# We only infer 2030 now
YEAR = "2030"
SSP_FOLDERS = [f"SSP{i}" for i in range(1, 6)]

# Apply sigmoid here if the model head has no activation
APPLY_SIGMOID = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float32

# Normalization stats (saved from training; see norm_stats.npz export)
NORM_STATS_PATH = "checkpoints/norm_stats.npz"  # .npz with 'mean' and 'std'
FALLBACK_MEAN = None  # optional manual override
FALLBACK_STD  = None

# Optional: cap extreme values like in training
CLAMP_MIN, CLAMP_MAX = -1e6, 1e6

# Batch tiles for speed (works if model is fully convolutional)
BATCH_SIZE = 4


In [4]:
# Cell 3 — Model loader (U-Net regression)

def load_model(in_channels: int) -> torch.nn.Module:
    """
    Rebuild the U-Net used during training and load weights from the checkpoint.
    """
    ckpt = torch.load(CKPT_PATH, map_location="cpu")

    # Try to recover backbone and input channels from checkpoint metadata
    backbone = ckpt.get("backbone", "resnet34")
    ckpt_in_ch = ckpt.get("in_channels", in_channels)

    model = smp.Unet(
        encoder_name=backbone,
        encoder_weights=None,      # no ImageNet pretraining; matches training
        in_channels=ckpt_in_ch,    # same channels as during training
        classes=1,
        activation=None,           # sigmoid applied externally if needed
    )

    # Your training code saved "model_state"
    if "model_state" in ckpt:
        state_dict = ckpt["model_state"]
    elif "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    else:
        state_dict = ckpt

    model.load_state_dict(state_dict, strict=True)
    model.to(DEVICE).eval()
    return model


In [5]:
# Cell 4 — Helper functions: file listing, normalization stats

def natural_key(s: str):
    """Sort strings with embedded numbers: class_2 < class_10."""
    return [int(t) if t.isdigit() else t.lower()
            for t in re.findall(r'\d+|\D+', os.path.basename(s))]

def list_2030_rasters(ssp_dir: str) -> List[str]:
    """
    Grab only 2030 rasters, sorted deterministically.
    Adjust this glob if needed to exclude labels etc.
    """
    paths = glob(os.path.join(ssp_dir, f"*{YEAR}*.tif"))
    # Example: to exclude CISI labels if present:
    # paths = [p for p in paths if "CISI" not in os.path.basename(p)]
    paths.sort(key=natural_key)
    if not paths:
        raise FileNotFoundError(f"No *{YEAR}*.tif found in {ssp_dir}")
    return paths

def load_norm_stats(n_channels: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load per-channel mean/std from norm_stats.npz.
    Falls back to zeros/ones if not available.
    """
    if os.path.isfile(NORM_STATS_PATH):
        stats = np.load(NORM_STATS_PATH)
        mean = stats["mean"].astype(np.float32)
        std  = stats["std"].astype(np.float32)
    else:
        if FALLBACK_MEAN is None or FALLBACK_STD is None:
            mean = np.zeros(n_channels, dtype=np.float32)
            std  = np.ones(n_channels, dtype=np.float32)
        else:
            mean = np.asarray(FALLBACK_MEAN, dtype=np.float32)
            std  = np.asarray(FALLBACK_STD,  dtype=np.float32)

    if mean.shape[0] != n_channels or std.shape[0] != n_channels:
        raise ValueError(
            f"Norm stats (C={mean.shape[0]}) do not match input channels (C={n_channels})."
        )

    # Guard against zero std
    std = np.where(std == 0, 1.0, std)
    return mean, std



In [9]:
# Cell 5 — Alignment checks and IO helpers (verbose)

def check_alignment(srcs: List[rasterio.DatasetReader]):
    """
    Ensure all input rasters share CRS, transform, width, and height.
    If not, raise with details.
    """
    base_ds = srcs[0]
    base_name = os.path.basename(base_ds.name)
    crs = base_ds.crs
    transform = base_ds.transform
    width, height = base_ds.width, base_ds.height

    for i, ds in enumerate(srcs[1:], start=1):
        name = os.path.basename(ds.name)
        problems = []
        if ds.crs != crs:
            problems.append("CRS")
        if ds.transform != transform:
            problems.append("transform")
        if ds.width != width or ds.height != height:
            problems.append("shape")

        if problems:
            raise ValueError(
                f"Input rasters are not aligned between '{base_name}' and '{name}' "
                f"(mismatch in: {', '.join(problems)})"
            )


In [7]:
# Cell 6 — Inference for a single SSP folder

def infer_one_ssp(ssp_path: str, out_path: str):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    # Discover 2030 predictor rasters for this SSP
    raster_paths = list_2030_rasters(ssp_path)
    n_channels = len(raster_paths)

    # Open sources and validate alignment
    srcs = [rasterio.open(p) for p in raster_paths]
    try:
        check_alignment(srcs)
        ref = srcs[0]
        profile = ref.profile.copy()
        profile.update(
            count=1,
            dtype="float32",
            nodata=None,
            compress="deflate",
            predictor=3,
            zlevel=6,
        )

        # Load normalization and model
        mean, std = load_norm_stats(n_channels)
        model = load_model(in_channels=n_channels)

        with rasterio.open(out_path, "w", **profile) as dst:
            tiles = []
            tile_windows = []

            # Iterate over internal raster block windows for efficient IO
            for _, window in ref.block_windows(1):
                x_np = read_stack_window(srcs, window)          # (C, h, w)
                tiles.append(torch.from_numpy(x_np).unsqueeze(0))  # (1, C, h, w)
                tile_windows.append(window)

                # Process in mini-batches
                if len(tiles) == BATCH_SIZE:
                    batch = torch.cat(tiles, dim=0).to(DEVICE, dtype=DTYPE)
                    batch = normalize_batch(batch, mean, std)
                    with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):
                        y = model(batch)                        # (B, 1, h, w)
                        if APPLY_SIGMOID:
                            y = torch.sigmoid(y)
                    y_np = y.squeeze(1).cpu().numpy().astype(np.float32)  # (B, h, w)

                    for arr, win in zip(y_np, tile_windows):
                        write_window(dst, win, arr)

                    tiles.clear()
                    tile_windows.clear()

            # Flush remaining tiles
            if tiles:
                batch = torch.cat(tiles, dim=0).to(DEVICE, dtype=DTYPE)
                batch = normalize_batch(batch, mean, std)
                with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):
                    y = model(batch)
                    if APPLY_SIGMOID:
                        y = torch.sigmoid(y)
                y_np = y.squeeze(1).cpu().numpy().astype(np.float32)
                for arr, win in zip(y_np, tile_windows):
                    write_window(dst, win, arr)

        print(f"[OK] Wrote {out_path}")

    finally:
        for ds in srcs:
            ds.close()


In [8]:
# Cell 7 — Main loop over SSP1..SSP5

def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    for ssp in SSP_FOLDERS:
        ssp_dir = os.path.join(DATA_ROOT, ssp)
        if not os.path.isdir(ssp_dir):
            print(f"[WARN] Skipping missing {ssp_dir}")
            continue
        out_file = os.path.join(OUT_DIR, f"CISI_{YEAR}_{ssp}.tif")
        infer_one_ssp(ssp_dir, out_file)

if __name__ == "__main__":
    main()


ValueError: Input rasters are not aligned: mismatch at index 1